# Coflect HILT Workflow (TensorFlow / Keras)

This notebook follows a standard deep learning experiment flow with Coflect integration:
1. Imports and reproducibility
2. Data loading and visualization
3. Model/loss/optimizer setup
4. Training
5. Evaluation


## 1) Imports and reproducibility


In [ ]:
from __future__ import annotations

import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import tensorflow as tf

from coflect.backends import TensorFlowAdapter
from coflect.modules.hilt.common.tf_model import build_tf_cnn

from coflect.modules.hilt.common.notebook_bridge import (
    NotebookBridgeConfig,
    NotebookHILTBridge,
    roi_norm_to_pixels,
)


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("tensorflow:", tf.__version__)
print("gpu available:", bool(tf.config.list_physical_devices("GPU")))


## 2) Data loading and visualization


In [ ]:
@dataclass(frozen=True)
class ExperimentConfig:
    batch_size: int = 64
    epochs: int = 3
    learning_rate: float = 1e-3
    image_size: int = 64
    num_classes: int = 2
    data_root: str = "./data"
    split: str = "train"
    download_data: bool = True
    snapshot_dir: str = "./snapshots_tf"
    snapshot_every: int = 50
    xai_every: int = 25


cfg = ExperimentConfig()
print(cfg)

tf.get_logger().setLevel("ERROR")

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
y_train = y_train.squeeze()
y_test = y_test.squeeze()

# Keep only CIFAR-10 cat (3) and dog (5), and remap to {0, 1}.
train_mask = np.logical_or(y_train == 3, y_train == 5)
test_mask = np.logical_or(y_test == 3, y_test == 5)

x_train = x_train[train_mask].astype("float32") / 255.0
x_test = x_test[test_mask].astype("float32") / 255.0
y_train = (y_train[train_mask] == 5).astype("int32")
y_test = (y_test[test_mask] == 5).astype("int32")

train_idx = np.arange(x_train.shape[0], dtype=np.int32)
train_batches = int(np.ceil(float(x_train.shape[0]) / float(cfg.batch_size)))

print("train samples:", x_train.shape[0], "val samples:", x_test.shape[0])

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train, train_idx))
    .shuffle(buffer_size=min(10_000, x_train.shape[0]), seed=SEED)
    .batch(cfg.batch_size)
    .map(
        lambda x, y, idx: (tf.image.resize(x, (cfg.image_size, cfg.image_size)), y, idx),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(cfg.batch_size)
    .map(
        lambda x, y: (tf.image.resize(x, (cfg.image_size, cfg.image_size)), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    .prefetch(tf.data.AUTOTUNE)
)

snapshot_dir = Path(cfg.snapshot_dir)
snapshot_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
axes = axes.flatten()
for i in range(8):
    img = x_train[i]
    label = int(y_train[i])
    class_name = "dog" if label == 1 else "cat"
    axes[i].imshow(img)
    axes[i].set_title(class_name)
    axes[i].axis("off")

plt.tight_layout()
plt.show()


## 3) Model, loss, optimizer, and Coflect adapter setup


In [ ]:
model = build_tf_cnn(num_classes=cfg.num_classes, image_size=cfg.image_size)
optimizer = tf.keras.optimizers.Adam(learning_rate=cfg.learning_rate)
adapter = TensorFlowAdapter(model=model, optimizer=optimizer)

worker_candidates = [
    Path("coflect_tf_xai_worker_local.py"),
    Path("examples/hilt/coflect_tf_xai_worker_local.py"),
    Path("HILTM/examples/hilt/coflect_tf_xai_worker_local.py"),
]
worker_script = next((str(p.resolve()) for p in worker_candidates if p.exists()), "")
print("xai worker script:", worker_script or "<package module>")

bridge = NotebookHILTBridge(
    NotebookBridgeConfig(
        server="http://127.0.0.1:8000",
        backend="tensorflow",
        metrics_every=10,
        feedback_every=10,
        xai_every=cfg.xai_every,
        auto_start_backend=True,
        auto_start_xai_worker=True,
        backend_host="127.0.0.1",
        backend_port=8000,
        xai_snapshot_dir=str(snapshot_dir),
        xai_dataset="cifar10_catsdogs",
        data_root=cfg.data_root,
        split=cfg.split,
        download_data=cfg.download_data,
        xai_num_classes=cfg.num_classes,
        xai_image_size=cfg.image_size,
        xai_method="consensus",
        xai_worker_script=worker_script,
        log_dir="./.coflect_logs/notebook_bridge",
    )
)

print(model.summary())
print("bridge auto-start enabled: backend + XAI worker should launch automatically")
print("open UI at http://127.0.0.1:8000")


## 4) Training

This loop emits compact metrics, periodic XAI requests, and async snapshots.
Live panel updates require backend + TF XAI worker (auto-started by bridge config).


In [ ]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)


def evaluate(model: tf.keras.Model, dataset: tf.data.Dataset) -> tuple[float, float]:
    val_loss = tf.keras.metrics.Mean()
    val_acc = tf.keras.metrics.SparseCategoricalAccuracy()

    for x_batch, y_batch in dataset:
        logits = model(x_batch, training=False)
        loss = loss_fn(y_batch, logits)
        val_loss.update_state(loss)
        val_acc.update_state(y_batch, logits)

    return float(val_loss.result().numpy()), float(val_acc.result().numpy())


In [ ]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "epoch_time_sec": [],
}

global_step = 0
focus_lambda = bridge.focus_lambda
roi_norm = bridge.roi_norm

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()

    train_loss_metric = tf.keras.metrics.Mean()
    train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
    epoch_step = 0

    for x_batch, y_batch, sample_idx_batch in train_ds:
        global_step += 1
        epoch_step += 1

        update = bridge.maybe_sync_feedback(step=global_step)
        if update.changed:
            focus_lambda = update.focus_lambda
            roi_norm = update.roi_norm

        while bridge.paused:
            time.sleep(0.25)
            pause_update = bridge.maybe_sync_feedback(step=global_step, force=True)
            if pause_update.changed:
                focus_lambda = pause_update.focus_lambda
                roi_norm = pause_update.roi_norm

        with tf.GradientTape() as tape:
            logits = adapter.forward(x_batch)
            primary_loss = adapter.loss(logits, y_batch)
            total_loss = primary_loss

            if focus_lambda > 0.0 and roi_norm is not None:
                pixels = roi_norm_to_pixels(
                    roi_norm,
                    height=int(x_batch.shape[1]),
                    width=int(x_batch.shape[2]),
                )
                if pixels is not None:
                    x0, y0, x1, y1 = pixels
                    mask = np.zeros((1, int(x_batch.shape[1]), int(x_batch.shape[2]), 1), dtype=np.float32)
                    mask[:, y0:y1, x0:x1, :] = 1.0
                    mask_t = tf.convert_to_tensor(mask, dtype=x_batch.dtype)
                    masked_x = x_batch * mask_t
                    aux_logits = adapter.forward(masked_x)
                    aux_loss = adapter.loss(aux_logits, y_batch)
                    total_loss = total_loss + float(focus_lambda) * aux_loss

        grads = tape.gradient(total_loss, adapter.model.trainable_variables)
        adapter.optimizer.apply_gradients(zip(grads, adapter.model.trainable_variables))

        preds = tf.argmax(logits, axis=1, output_type=tf.int32)
        y_int = tf.cast(y_batch, tf.int32)
        batch_acc = tf.reduce_mean(tf.cast(tf.equal(preds, y_int), tf.float32))

        train_loss_metric.update_state(primary_loss)
        train_acc_metric.update_state(y_batch, logits)

        sample_idx0 = int(sample_idx_batch[0].numpy())
        target0 = int(y_int[0].numpy())
        pred0 = int(preds[0].numpy())
        bridge.maybe_enqueue_xai(
            step=global_step,
            sample_idx=sample_idx0,
            target_class=target0,
            pred_class=pred0,
            request_kind="periodic",
            force=(global_step == 1),
        )

        if global_step == 1 or global_step % max(1, cfg.snapshot_every) == 0:
            tmp_path = snapshot_dir / f".model_step_{global_step:06d}.npz.tmp"
            final_path = snapshot_dir / f"model_step_{global_step:06d}.npz"
            with open(tmp_path, "wb") as fh:
                np.savez(fh, *model.get_weights())
            tmp_path.replace(final_path)

        bridge.maybe_post_metrics(
            step=global_step,
            loss=float(primary_loss.numpy()),
            acc=float(batch_acc.numpy()),
            epoch=float(epoch - 1) + (float(epoch_step) / max(1.0, float(train_batches))),
            focus_lambda=float(focus_lambda),
        )

    train_loss = float(train_loss_metric.result().numpy())
    train_acc = float(train_acc_metric.result().numpy())
    val_loss, val_acc = evaluate(model, val_ds)
    dt = time.time() - t0

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["epoch_time_sec"].append(dt)

    bridge.maybe_post_metrics(
        step=global_step,
        loss=float(train_loss),
        acc=float(train_acc),
        epoch=float(epoch),
        focus_lambda=float(focus_lambda),
        extra={"val_loss": float(val_loss), "val_acc": float(val_acc)},
        force=True,
    )

    print(
        f"epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"focus={focus_lambda:.3f} | "
        f"time={dt:.1f}s"
    )


## 5) Evaluation and plots


In [ ]:
epochs = np.arange(1, cfg.epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(epochs, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs, history["val_loss"], marker="o", label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs, history["val_acc"], marker="o", label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
final_val_loss, final_val_acc = evaluate(model, val_ds)
print(f"final validation loss: {final_val_loss:.4f}")
print(f"final validation accuracy: {final_val_acc:.4f}")
